# RouterGuard — Step 3: calibration-time balancing, baselines, accuracy-vs-compression

**Gate for Step 3:** an accuracy-vs-compression curve exists, and the balancing decision is
recorded against the numeric gates (`max_kl_nats`, `min_balance_transfer_ratio`). Step 1 already
decided that balancing is worth attempting (calib Gini 0.271 > 0.20).

What this notebook does, in order:

1. baseline metrics on the unpruned model (ppl, MMLU, throughput, memory)
2. fit per-expert router biases on **calibration data only**, freeze them, then re-measure load
   statistics on **both** splits
3. check the two balancing guards, and measure what the bias costs in perplexity
4. frequency and random pruning baselines at 25% and 50%, evaluated via mask simulation
   (bitwise-equivalent to real removal, verified in Step 2) so no reloads are needed
5. one *real* removal at 50% purely to measure the memory actually saved

Runtime on a T4 with 8-bit OLMoE: roughly 25–35 minutes. Set `RUN_MMLU_ON_CURVE = True` only if
you have the time budget — it adds ~4x the curve cost.

In [ ]:
# 00 — install pinned stack (skip if notebook 01 already did it and you restarted).
# The huggingface-hub and datasets bounds are mandatory: a newer `datasets` upgrades
# huggingface-hub past 1.0 as a side effect, which makes transformers 4.48.3
# unimportable. env.assert_env() in the next cell enforces the same range.
INSTALL = False

if INSTALL:
    import subprocess, sys
    pkgs = [
        "transformers==4.48.3",
        "huggingface-hub>=0.24,<1.0",
        "datasets>=2.19,<5",
        "accelerate>=0.34",
        "bitsandbytes>=0.43.3",
        "pyyaml",
        "matplotlib",
        "sentencepiece",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    print("installed:", " ".join(pkgs))
    print("RESTART THE RUNTIME before continuing.")

In [ ]:
# 01 — bootstrap
import os, sys, json, gc
from pathlib import Path

ROOT = Path(os.environ.get("ROUTERGUARD_ROOT", "/content/routerguard")).expanduser()
if not (ROOT / "routerguard" / "env.py").exists():
    for candidate in (Path.home() / "routerguard", Path.cwd(), Path.cwd().parent):
        if (candidate / "routerguard" / "env.py").exists():
            ROOT = candidate
            break
if not (ROOT / "routerguard" / "env.py").exists():
    raise SystemExit("routerguard not found; set ROUTERGUARD_ROOT or clone the repo")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

RUN_MMLU_ON_CURVE = False   # keep False unless you have runtime to spare

In [ ]:
# 02 — environment, run bookkeeping, protocol
import torch
from routerguard import env, models, data, stats, moe, balance, prune, eval as rgeval, run as rgrun

report = env.assert_env(strict=True)
print(f"transformers {report['versions']['transformers']} (primary pin: {report['transformers_is_primary']})")
print(f"gpu {report['gpu']['name']} ({report['gpu']['total_gb']} GB, cc {report['gpu']['compute_capability']})")

PROTOCOL = rgrun.load_protocol(ROOT / "configs" / "protocol.yaml")
RUN = rgrun.new_run("step3_balance", protocol=PROTOCOL)
rgrun.set_seed(RUN.seed)
print(RUN)

In [ ]:
# 03 — model + splits + MMLU prompts (splits cached in a stable dir so every run reuses them)
model, tokenizer, info = models.load_olmoe(precision="auto")
print(info)
RUN.save_json("load_info", info.to_dict())

splits = data.build_splits(tokenizer, PROTOCOL, cache_dir=ROOT / "results" / "splits")
for name, split in splits.items():
    print(split.describe())

mmlu_cfg = PROTOCOL["metrics"]["mmlu"]
prompts_path = ROOT / "results" / f"mmlu_{mmlu_cfg['n_questions']}_{mmlu_cfg['n_shot']}shot_seed{mmlu_cfg['seed']}.json"
if prompts_path.exists():
    prompts = json.loads(prompts_path.read_text())
    print(f"mmlu: loaded {len(prompts)} cached prompts")
else:
    prompts = data.build_mmlu_prompts(
        n_questions=int(mmlu_cfg["n_questions"]), n_shot=int(mmlu_cfg["n_shot"]), seed=int(mmlu_cfg["seed"])
    )
    prompts_path.parent.mkdir(parents=True, exist_ok=True)
    prompts_path.write_text(json.dumps(prompts))
    print(f"mmlu: built {len(prompts)} prompts -> {prompts_path}")
print("subjects:", len({p["subject"] for p in prompts}))

In [ ]:
# 04 — baseline metrics on the unpruned model
calib_ids = splits["calib"].input_ids.to("cuda")
eval_ids = splits["eval"].input_ids.to("cuda")

baseline = rgeval.evaluate_all(
    model, tokenizer, eval_ids=eval_ids, prompts=prompts, protocol=PROTOCOL,
    precision=info.precision, device="cuda", batch_size=1,
    mmlu_batch_size=8, do_throughput=True,
)
print(f"perplexity : {baseline['perplexity']['perplexity']:.4f}  ({baseline['perplexity']['tokens']} tokens)")
print(f"mmlu       : {baseline['mmlu']['accuracy']:.4f}  ({baseline['mmlu']['n_shot']}-shot, {baseline['mmlu']['n_questions']} q)")
print(f"throughput : {baseline['throughput']['tokens_per_sec_mean']:.1f} tok/s (cv {baseline['throughput']['cv']:.3f})")
print(f"memory     : {baseline['memory'].get('allocated_gb')} GB allocated")
RUN.save_json("metrics_baseline", baseline)

calib_stats = balance.measure_model(model, calib_ids, None, batch_size=4, device="cuda")
print(f"\ncalib gini_mean = {calib_stats['gini_mean']:.4f}")
RUN.save_json("load_stats_calib_unbalanced", calib_stats)

In [ ]:
# 05 — fit calibration-time biases (weights frozen), then re-measure on calib AND eval
result = balance.evaluate_balance(
    model, calib_ids, eval_ids, PROTOCOL, batch_size=4, device="cuda", progress=True
)
print()
print(f"u={result.u} iters={result.iters}  max|bias|={result.max_abs_bias:.4f}")
print(f"gini calib : {result.calib_before['gini_mean']:.4f} -> {result.calib_after['gini_mean']:.4f}  (gain {result.gini_gain_calib:+.4f})")
print(f"gini eval  : {result.eval_before['gini_mean']:.4f} -> {result.eval_after['gini_mean']:.4f}  (gain {result.gini_gain_eval:+.4f})")
print(f"transfer   : {result.transfer_ratio:.3f}   KL mean {result.kl_mean:.5f} nats, max {result.kl_max:.5f}")
print(f"max-viol   : calib {result.calib_before['max_violation_max']:.3f} -> {result.calib_after['max_violation_max']:.3f}")
for w in result.warnings():
    print(f"WARNING: {w}")

summary = result.summary()
summary["gates"] = rgrun.summarize_gates(PROTOCOL, result.gate_inputs())
RUN.save_json("balance", summary)
result.save(RUN.dir / "balance_biases.pt")
print()
print(rgrun.format_gates(summary["gates"]))

In [ ]:
# 06 — what does the bias cost in accuracy? (balancing must not buy Gini with ppl)
# The unpruned, unbiased perplexity is already in `baseline`, so only the biased pass runs.
ppl_plain = baseline["perplexity"]["perplexity"]
with torch.no_grad(), balance.apply_bias(model, result.biases, device="cuda"):
    ppl_with_bias = rgeval.wikitext_perplexity(model, eval_ids, batch_size=1, device="cuda")

delta = ppl_with_bias["perplexity"] - ppl_plain
print(f"ppl without bias : {ppl_plain:.4f}")
print(f"ppl with bias    : {ppl_with_bias['perplexity']:.4f}   (delta {delta:+.4f})")
RUN.save_json(
    "balance_ppl_cost",
    {"without_bias_ppl": ppl_plain, "with_bias": ppl_with_bias, "delta": delta},
)

In [ ]:
# 07 — plots: per-layer Gini before/after, and utilization for the worst layer
import matplotlib.pyplot as plt

layers = sorted(result.calib_before["per_layer_gini"])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for report, style, label in (
    (result.calib_before, "o-", "calib before"),
    (result.calib_after, "s-", "calib after"),
    (result.eval_before, "^--", "eval before"),
    (result.eval_after, "v--", "eval after"),
):
    axes[0].plot(layers, [report["per_layer_gini"][layer] for layer in layers], style, label=label)
axes[0].axhline(PROTOCOL["gates"]["gini_skip_balance"], color="red", ls=":", label="gate")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("Gini of expert load")
axes[0].set_title("Calibration-time balancing: load imbalance")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

worst = max(result.calib_before["per_layer_gini"], key=lambda k: result.calib_before["per_layer_gini"][k])
worst = int(worst)
before = torch.tensor(result.calib_before["per_layer_counts"][worst], dtype=torch.float32)
after = torch.tensor(result.calib_after["per_layer_counts"][worst], dtype=torch.float32)
x = range(len(before))
axes[1].bar([i - 0.2 for i in x], (before / before.mean()).numpy(), width=0.4, label="before")
axes[1].bar([i + 0.2 for i in x], (after / after.mean()).numpy(), width=0.4, label="after")
axes[1].axhline(1.0, color="black", lw=0.8)
axes[1].set_xlabel("expert index"); axes[1].set_ylabel("load / uniform")
axes[1].set_title(f"Layer {worst}: per-expert utilization"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RUN.path("figures", "step3_balance.png"), dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# 08 — accuracy-vs-compression curve for the baselines, via mask simulation.
# MaskSim is bitwise-equivalent to real removal at this precision (Step 2), so we can
# evaluate several compression points on one loaded model.
experts = prune.experts_per_layer(model)
calib_counts = calib_stats["per_layer_counts"]
ratios = [float(r) for r in PROTOCOL["pruning"]["ratios"]]
seeds = [int(s) for s in PROTOCOL["pruning"]["seeds"]]

curve = []
for kind in PROTOCOL["design"]["baselines"]:
    for ratio in ratios:
        keeps = prune.keeps_for(model, kind, ratio, per_layer_counts=calib_counts, seed=seeds[0])
        with torch.no_grad(), moe.MaskSim(model, keeps):
            ppl = rgeval.wikitext_perplexity(model, eval_ids, batch_size=1, device="cuda")
            point = {
                "kind": kind, "ratio": ratio, "seed": seeds[0],
                "kept_per_layer": [len(v) for v in keeps.values()][:1],
                "perplexity": ppl["perplexity"], "tokens": ppl["tokens"],
            }
            if RUN_MMLU_ON_CURVE:
                mmlu = rgeval.mmlu_accuracy(model, tokenizer, prompts, batch_size=8, device="cuda")
                point["mmlu"] = mmlu["accuracy"]
        curve.append(point)
        print(f"{kind:9} ratio={ratio:.2f}  kept/layer={point['kept_per_layer'][0]:3d}  ppl={point['perplexity']:.4f}"
              + (f"  mmlu={point.get('mmlu', float('nan')):.4f}" if RUN_MMLU_ON_CURVE else ""))

RUN.save_json("curve_baselines", {"baseline_ppl": baseline["perplexity"]["perplexity"], "points": curve})

fig, ax = plt.subplots(figsize=(6, 4))
for kind in PROTOCOL["design"]["baselines"]:
    xs = [0.0] + [p["ratio"] * 100 for p in curve if p["kind"] == kind]
    ys = [baseline["perplexity"]["perplexity"]] + [p["perplexity"] for p in curve if p["kind"] == kind]
    ax.plot(xs, ys, "o-", label=kind)
ax.set_xlabel("routed experts removed (%)"); ax.set_ylabel("wikitext perplexity")
ax.set_title("Accuracy vs compression (baselines)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RUN.path("figures", "step3_curve.png"), dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# 09 — memory actually saved by a real 50% removal (this mutates the model; do it last)
gc.collect(); torch.cuda.empty_cache()
mem_before = rgeval.memory_snapshot()
params_before = rgeval.param_report(model, info.precision)

keeps_50 = prune.keeps_for(model, "frequency", 0.50, per_layer_counts=calib_counts, seed=seeds[0])
removal = prune.prune_with_keeps(model, keeps_50)

gc.collect(); torch.cuda.empty_cache()
mem_after = rgeval.memory_snapshot()
params_after = rgeval.param_report(model, info.precision)

print(f"experts/layer : {removal['num_experts'][0]}  (removed {removal['removed_total']} total)")
print(f"params        : {params_before['params_total']:,} -> {params_after['params_total']:,}")
print(f"measured alloc: {mem_before.get('allocated_gb')} GB -> {mem_after.get('allocated_gb')} GB")
print(f"estimated     : {params_before['estimated_gb']} GB -> {params_after['estimated_gb']} GB")

memory_report = {
    "removal": {k: v for k, v in removal.items() if k != "layers"},
    "params_before": params_before, "params_after": params_after,
    "measured_before": mem_before, "measured_after": mem_after,
}
RUN.save_json("memory_saved_50pct", memory_report)

with torch.no_grad():
    ppl_pruned = rgeval.wikitext_perplexity(model, eval_ids, batch_size=1, device="cuda")
print(f"ppl after real removal: {ppl_pruned['perplexity']:.4f} (mask sim said {[p['perplexity'] for p in curve if p['kind']=='frequency' and p['ratio']==0.5]})")

## Step 3 gate

Record before moving to Step 4:

- [ ] baseline ppl / MMLU / throughput / memory saved for the unpruned model
- [ ] balancing: Gini gain on calib **and** eval, with the transfer ratio
- [ ] `max_kl_nats` pass/fail (measured per-layer KL on eval)
- [ ] `min_balance_transfer_ratio` pass/fail
- [ ] ppl cost of the bias recorded (balancing must not buy Gini with accuracy)
- [ ] accuracy-vs-compression curve for frequency and random baselines
- [ ] memory actually saved at 50% removal (headline metric)

Two outcomes that change Step 4:

* **Overshoot warning fired** (`u` too large, Gini rose): reduce `u` in `protocol.yaml`, note the
  change, and re-run cell 05. The sign update has a real instability boundary.
* **Transfer ratio < 0.5 or KL > 0.05**: the calibration-fitted bias does not generalize, which is a
  *negative result about the balancing component* and must be reported, not tuned away.

Step 4 replaces the frequency baseline with the redundancy-aware ranking (`similarity.py` + greedy
pruning) and runs the 2x2 ablation: balance on/off x redundancy on/off.